# Data Collection - SpaceX API

The raw response is the historical SpaceX API snapshot supplied by IBM. The normalized Falcon 9 checkpoint supplies the modeling features. This preserves the course time window rather than mixing in current launches.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def collect_api():
    if not (DATA/'API_call_spacex_api.json').exists():
        import urllib.request
        url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
        # GET the course archive of the SpaceX v4 launches/past response.
        (DATA/'API_call_spacex_api.json').write_bytes(urllib.request.urlopen(url,timeout=60).read())
    raw=json.loads((DATA/'API_call_spacex_api.json').read_text())
    rows=[]
    for r in raw:
        core=(r.get('cores') or [{}])[0]
        rows.append({'APIFlightNumber':r['flight_number'],'Date':r['date_utc'][:10],
          'Name':r['name'],'RocketId':r['rocket'],'LaunchpadId':r['launchpad'],
          'PayloadIds':','.join(r['payloads']), 'LandingSuccess':core.get('landing_success'),
          'LandingType':core.get('landing_type'),'CoreId':core.get('core'),
          'Reused':core.get('reused'),'Flight':core.get('flight')})
    api=pd.DataFrame(rows); api.to_csv(DATA/'api_launches.csv',index=False)
    print('SpaceX API historical snapshot:',len(api),'launches')
    print(api.head().to_string(index=False))
    return api


In [2]:
api = collect_api()

SpaceX API historical snapshot: 107 launches
 APIFlightNumber       Date        Name                 RocketId              LaunchpadId                                        PayloadIds LandingSuccess LandingType                   CoreId  Reused  Flight
               1 2006-03-24   FalconSat 5e9d0d95eda69955f709d1eb 5e9e4502f5090995de566f86                          5eb0e4b5b6c3bb0006eeb1e1           None         NaN 5e9e289df35918033d3b2623   False       1
               2 2007-03-21     DemoSat 5e9d0d95eda69955f709d1eb 5e9e4502f5090995de566f86                          5eb0e4b6b6c3bb0006eeb1e2           None         NaN 5e9e289ef35918416a3b2624   False       1
               3 2008-08-03 Trailblazer 5e9d0d95eda69955f709d1eb 5e9e4502f5090995de566f86 5eb0e4b6b6c3bb0006eeb1e3,5eb0e4b6b6c3bb0006eeb1e4           None         NaN 5e9e289ef3591814873b2625   False       1
               4 2008-09-28      RatSat 5e9d0d95eda69955f709d1eb 5e9e4502f5090995de566f86                          5eb0e4b7